# 4. Building and training graph neural networks

## Goal

Convert SMILES into PyTorch Geometric graphs, inspect atom and bond
features, construct GATv2Conv or NNConv multi-head models, train joint or
single-task cross-validation folds, and save portable checkpoints.

## Setup

Install `.[notebook,training,gnn]`. The graph construction and forward
pass execute on CPU; the full 300-epoch CV fit is disabled by default.

### 1. Select graph features and targets

**Caution.** Graph featurisation must match between training and inference; changing explicit hydrogens or excluded attributes invalidates an existing checkpoint.

**Extension.** Ablation studies can pass atom or bond feature names through `exclude` whilst retaining the same training code.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
EXAMPLE_DATA = PROJECT_ROOT / "examples" / "data"
MODEL_DIRECTORY = PROJECT_ROOT / "models"
RESULT_DIRECTORY = PROJECT_ROOT / "results"
RESULT_DIRECTORY.mkdir(parents=True, exist_ok=True)

import pandas as pd
import torch

from lc_qspr import PHASE_TARGETS, TEMPERATURE_TARGETS
from lc_qspr.preprocessing import read_table, separate_chiral_phases

RUN_GNN_TRAINING = False
table, _ = separate_chiral_phases(read_table(EXAMPLE_DATA / "sample_desc.csv"))
phase_targets = [target for target in PHASE_TARGETS if target in table]
temperature_targets = [target for target in TEMPERATURE_TARGETS if target in table]
atom_encoding = "none"
excluded_attributes = ()
explicit_hydrogens = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


### 2. Build the in-memory graph dataset

**Caution.** The in-memory representation is convenient but can exhaust RAM for a very large database; measure a subset before scaling.

**Extension.** A disk-backed PyG dataset can reuse `smiles_to_graph` if the database no longer fits comfortably in memory.

In [2]:
from lc_qspr.gnn import LiquidCrystalGraphDataset

graph_dataset = LiquidCrystalGraphDataset(
    table["SMILES"],
    classification_targets=table[phase_targets],
    regression_targets=table[temperature_targets],
    atom_encoding=atom_encoding,
    exclude=excluded_attributes,
    explicit_hydrogens=explicit_hydrogens,
)
print("Graphs:", len(graph_dataset))
print("Node features:", graph_dataset.dim_node_attr)
print("Edge features:", graph_dataset.dim_edge_attr)
print("Output channels:", graph_dataset.output_channels)

Graphs: 5
Node features: 44
Edge features: 10
Output channels: [22, 24]


### 3. Run a model forward pass

**Caution.** Hidden channels must be divisible by the number of attention heads for GATv2Conv. The distributed checkpoint uses `mean-max` pooling without residual connections.

**Extension.** Set `pooling='mean-max-size'` to expose molecular size explicitly, or set `residual=True` to test a deep skip-connected model. These choices require fresh training.

In [3]:
from torch_geometric.loader import DataLoader
from lc_qspr.gnn import GNN

model = GNN(
    hidden_channels=32,
    num_layers=2,
    out_channels=graph_dataset.output_channels,
    dim_node_attr=graph_dataset.dim_node_attr,
    dim_edge_attr=graph_dataset.dim_edge_attr,
    num_heads=4,
    conv_type="GATv2Conv",
    pooling="mean-max",
    residual=False,
)
batch = next(iter(DataLoader(graph_dataset, batch_size=len(graph_dataset))))
outputs = model(batch)
print([tuple(output.shape) for output in outputs])

[(5, 22), (5, 24)]


### 4. Configure joint cross-validated training

**Caution.** The paper-scale setting is computationally intensive and should be run only after a small one-epoch smoke test.

**Extension.** Set `mode` to `classification` or `regression` to reproduce the single-task variants; `both` performs joint learning.

In [4]:
from lc_qspr.evaluation import cross_validation_splits
from lc_qspr.gnn import GNNTrainingConfig, train_gnn_cross_validation

gnn_result = None
if RUN_GNN_TRAINING:
    splits = cross_validation_splits(
        table, phase_targets,
        mode="classification", n_splits=5, random_state=42,
    )
    gnn_result = train_gnn_cross_validation(
        graph_dataset,
        splits,
        phase_targets=phase_targets,
        temperature_targets=temperature_targets,
        mode="both",
        config=GNNTrainingConfig(
            convolution="GATv2Conv",
            pooling="mean-max",
            residual=False,
            hidden_channels=256,
            num_heads=4,
            num_layers=12,
            dropout=0.5,
            batch_size=128,
            learning_rate=1e-3,
            epochs=300,
            warmup_epochs=10,
        ),
        device=device,
        checkpoint_directory=RESULT_DIRECTORY / "gnn_checkpoints",
    )
    gnn_result.predictions.to_csv(
        RESULT_DIRECTORY / "GNN_joint_oof.csv", index=True
    )
else:
    print("Full GNN training is disabled; graph construction and the forward pass were checked above.")

Full GNN training is disabled; graph construction and the forward pass were checked above.


### 5. Plot loss, accuracy, F1 and MAE histories

**Caution.** Training metrics are optimistic; prioritise validation curves and out-of-fold molecule-level errors.

**Extension.** Select another `target_index` to inspect a rare phase or a particular transition temperature.

In [5]:
from lc_qspr.plotting import plot_gnn_histories
from lc_qspr.evaluation import classification_summary, temperature_summary
from IPython.display import display

if gnn_result is not None:
    plot_gnn_histories(gnn_result.histories, metric="loss")
    plot_gnn_histories(gnn_result.histories, metric="f1", target_index=0)
    plot_gnn_histories(gnn_result.histories, metric="mae", target_index=0)
    display(classification_summary(gnn_result.predictions, phase_targets))
    display(temperature_summary(gnn_result.predictions, temperature_targets))

### 6. Load the distributed checkpoint

**Caution.** A checkpoint is meaningful only with its original graph featurisation and ordered target vocabulary.

**Extension.** `load_gnn` also accepts legacy full-module checkpoints and can re-save them in the portable state-dictionary format via `save_gnn`.

In [6]:
from lc_qspr.gnn import load_gnn

pretrained = load_gnn(
    MODEL_DIRECTORY / "GATv2Conv_phase_temp_cv0.pt",
    map_location="cpu",
)
print(pretrained.conv_type, len(pretrained.convs), [layer.out_features for layer in pretrained.fc2])

GATv2Conv 12 [34, 26, 26]
